In [1]:
import jax.numpy as jnp
from diffPLOG2TROE.kinetic_constants import Arrhenius, refit_arrhenius
from diffPLOG2TROE.physical_constants import constants

In [2]:
def plog_refit(P, plog):
    T = jnp.linspace(300, 2500, 100)
    conc = P / (constants.R_L_atm_K_mol * T) * jnp.float64(0.001)
    k_plog = plog.kinetic_constant(T)

    # Calculate rate constant k0 = k / [M] (divide by M concentration)
    k0_values = k_plog / conc  # [cm3/mol/s]

    # Perform 3-parameter fit on the temperature-dependent k0 values
    lnA_new, n_new, EaR_new = refit_arrhenius(k0_values, T, True)
    return jnp.exp(lnA_new), n_new, EaR_new * constants.R_cal_mol

In [3]:
P = 0.1     # Pressure [atm]
A = 3.442e+17  # Pre-exponential factor [cm6/mol2/s]
alfa = -2.748 # Temperature exponent [-]
Eact = -750.78 # Activation energy [cal/mol]
original_constant = Arrhenius(name="PLOG @ 0.1 atm", params=jnp.array([A, alfa, Eact]))

# Perform refitting
A_new, n_new, EA_new = plog_refit(P, original_constant)
refitted_constant = Arrhenius(name="k0 refitted", params=jnp.array([A_new, n_new, EA_new]))

# Print results
print(f"\nPLOG Rate Refitting Results at P = {P} atm:")
print("==========================================")
print(original_constant)
print(refitted_constant)

print("\nVerification at selected temperatures:")
print(" Temperature [K] | [M] [mol/cm3]  | k0 [cm3/mol/s] | k0*[M] [cm6/mol2/s]")
print("-----------------|----------------|----------------|--------------------")
for T in [300, 1000, 2000]:
    conc = P / (constants.R_L_atm_K_mol * T) * jnp.float64(0.001)
    k0_at_T = refitted_constant.kinetic_constant(T)
    k_plog_at_T = original_constant.kinetic_constant(T)
    print(f"{T:15.1f}  | {conc:14.4e} | {k0_at_T:14.4e} | {k0_at_T * conc:16.4e}")


PLOG Rate Refitting Results at P = 0.1 atm:
PLOG @ 0.1 atm		3.44200e+17 -2.74800e+00 -7.50780e+02
k0 refitted		2.82441e+20 -1.74800e+00 -7.50780e+02

Verification at selected temperatures:
 Temperature [K] | [M] [mol/cm3]  | k0 [cm3/mol/s] | k0*[M] [cm6/mol2/s]
-----------------|----------------|----------------|--------------------
          300.0  |     4.0622e-06 |     4.6543e+16 |       1.8907e+11
         1000.0  |     1.2187e-06 |     2.3497e+15 |       2.8635e+09
         2000.0  |     6.0933e-07 |     5.7912e+14 |       3.5287e+08
